# 08 · Comparar los checkpoints de **dos** corridas de gatos

Compara los snapshots de **dos fuentes** a la vez y grafica las curvas de los dos experimentos:

| fuente | config | `num_steps` | salida |
|---|---|---|---|
| `base` | `config/cats_vp.yaml` | 60 000 | `…/phase_2/` |
| `overfit` | `config/cats_vp-overfitting.yaml` | 120 000 | `…/phase_2-overfitting/` |

### Lo primero que hay que saber: las dos configs son la misma, salvo el horizonte

Diffeadas clave por clave, difieren en **4 de 38**: `train.num_steps` (60 000 vs 120 000) y las tres rutas de `out`. Mismo
`seed`, mismo dataset y split, misma U-Net, mismo `lr`, mismo `ema_decay`, mismo `time_sampling`, mismo `amp`.

Eso no es un detalle administrativo, cambia cómo se lee todo el notebook:

- **El eje de pasos es directamente comparable.** Las dos curvas van en el mismo gráfico y la de `overfit` **prolonga** la de
  `base`; no son dos experimentos que haya que normalizar antes de superponer.
- **Los 8 checkpoints forman una sola progresión.** Con `checkpoint_every: 15000` y `keep_last_checkpoints: 4`, `base`
  conserva 15k/30k/45k + el final de 60k, y `overfit` conserva 60k/75k/90k/105k + el final de 120k. Tomando los **4 más
  avanzados de cada fuente** salen `15k, 30k, 45k, 60k` y `75k, 90k, 105k, 120k`: ocho pasos consecutivos, sin repetir ninguno.
- **Donde se solapan, deberían coincidir.** El snapshot de 60k de `overfit` y el final de 60k de `base` son, en teoría, el
  mismo modelo. Si difieren, la diferencia es no-determinismo (orden de carga con `num_workers: 8`, AMP, `cudnn.benchmark`), no
  entrenamiento. El notebook lo chequea solo y lo reporta: es una verificación gratis que sale de tener las dos corridas, y da
  la resolución con la que se pueden leer las demás diferencias.
- **Lo que el par 60k↔120k está para mostrar es el sobreajuste**, así que la figura que importa no es la pérdida sino el
  **gap** `val − train(examen fijo)` en función del paso.

### La trampa de las rutas

Las dos corridas escriben el **mismo basename**: `cats_vp.pt`, `cats_vp_stepNNNNN.pt`, `cats_vp_train_log.jsonl`. Solo cambia
el directorio. Por eso **todo se indexa por `(fuente, paso)`** y nunca por el paso solo — con un dict indexado por paso, la
segunda corrida le pisaría las muestras a la primera en silencio.

### Consideraciones de descubrimiento (siguen valiendo)

- Al lado de cada `<stem>_step{paso:05d}.pt` vive su sidecar `<stem>_step{paso:05d}.resume.pt`. Un glob ingenuo lo matchea y
  **no** es un checkpoint de pesos → se excluye explícitamente.
- El **paso** sale de `len(meta["history"])` (un valor por paso), más confiable que parsear el nombre y también válido para el
  checkpoint final, que no lleva sufijo.
- Cada fuente acepta override por entorno (`CATS_VP_CKPT`/`CATS_VP_LOG` y `CATS_VP_OVERFIT_CKPT`/`CATS_VP_OVERFIT_LOG`) para
  correr el notebook en otra máquina sin editar celdas.

Para el estudio del **Eje 2** (los 4 samplers sobre *un* checkpoint, a NFE igualado) el notebook es el `07`.

In [ ]:
# --- Setup: bootstrap de sys.path + imports + helpers ---
import sys
import pathlib
import os
import re
import json
import time
from collections import namedtuple

_here = pathlib.Path.cwd()
_root = None
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "diffusion").is_dir():
        _root = _cand
        break
if _root is None:
    raise RuntimeError(f"No encontré src/diffusion subiendo desde {_here}")
_src = str((_root / "src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)

import numpy as np
import torch
import matplotlib.pyplot as plt

from diffusion.training import load_checkpoint
from diffusion.samplers import generate_from_checkpoint, available_samplers

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def denorm(t):
    """(B,3,H,W) en [-1,1] -> (B,H,W,3) numpy en [0,1]."""
    x = (t.detach() * 0.5 + 0.5).clamp(0.0, 1.0)
    return x.permute(0, 2, 3, 1).cpu().numpy()


print("paquete en:", _root)
print("torch:", torch.__version__, "| device:", DEVICE)
print("samplers:", available_samplers())

# --- Presupuesto de computo: NFE, no pasos --------------------------------------
# Cada sampler gasta un numero DISTINTO de evaluaciones del score por paso (contado
# sobre el codigo de diffusion/samplers):
#   euler   -> 1                (_reverse_drift)
#   pf_ode  -> 1                (_pfode_drift)
#   heun    -> 2                (drift en t y en el estado predicho a t+dt)
#   pc      -> 1 + n_corrector  (predictor + cada correccion de Langevin)
# Darles el mismo `n_steps` NO es darles el mismo computo: heun y pc gastan el doble.
# Con la U-Net dominando el costo, el NFE es basicamente el tiempo de pared, asi que
# comparar samplers a `n_steps` igualado le regala 2x a dos de los cuatro.
N_CORRECTOR = 1
NFE_POR_PASO = {"euler": 1, "pf_ode": 1, "heun": 2, "pc": 1 + N_CORRECTOR}


def pasos_para(nom_sampler, nfe):
    """Pasos de integracion que gastan `nfe` evaluaciones de score con este sampler."""
    if nom_sampler not in NFE_POR_PASO:
        raise KeyError(f"NFE/paso desconocido para '{nom_sampler}'; agregalo a NFE_POR_PASO")
    return max(1, nfe // NFE_POR_PASO[nom_sampler])


def kwargs_para(nom_sampler):
    """Kwargs propios del sampler. Solo `pc` los tiene, y `n_corrector` va explicito
    porque el conteo de NFE depende de el: si cambia el default, el conteo miente."""
    return {"n_corrector": N_CORRECTOR} if nom_sampler == "pc" else {}


print("NFE por paso:", NFE_POR_PASO)

## 1. Resolver las rutas de las dos fuentes

Cada fuente sale de su YAML (`out.checkpoint` / `out.train_log`), con override por variable de entorno. Se verifica que las
dos **no** apunten al mismo lugar: como comparten el basename `cats_vp`, dos configs mal resueltas darían el mismo conjunto de
archivos descubierto dos veces, y el notebook "compararía" una corrida contra sí misma sin avisar.

In [ ]:
# --- Las dos fuentes ---
import yaml

FUENTES = [
    {"clave": "base",    "config": "cats_vp.yaml",            "color": "C0", "env": "CATS_VP"},
    {"clave": "overfit", "config": "cats_vp-overfitting.yaml", "color": "C3", "env": "CATS_VP_OVERFIT"},
]

for f in FUENTES:
    cfg_path = _root / "config" / f["config"]
    ckpt_target = log_target = num_steps = None
    if cfg_path.exists():
        with open(cfg_path, "r", encoding="utf-8") as fh:
            _cfg = yaml.safe_load(fh)
        ckpt_target = pathlib.Path(_cfg["out"]["checkpoint"])
        if _cfg["out"].get("train_log"):
            log_target = pathlib.Path(_cfg["out"]["train_log"])
        num_steps = (_cfg.get("train") or {}).get("num_steps")

    # Override por entorno: para correr en otra máquina sin editar el notebook.
    if os.environ.get(f"{f['env']}_CKPT"):
        ckpt_target = pathlib.Path(os.environ[f"{f['env']}_CKPT"])
    if os.environ.get(f"{f['env']}_LOG"):
        log_target = pathlib.Path(os.environ[f"{f['env']}_LOG"])

    if ckpt_target is None:
        raise FileNotFoundError(
            f"Fuente '{f['clave']}': no pude resolver el checkpoint. Falta {cfg_path} "
            f"y no está {f['env']}_CKPT en el entorno."
        )

    f["ckpt_dir"] = ckpt_target.parent
    f["stem"] = ckpt_target.stem
    f["log"] = log_target
    f["num_steps"] = num_steps

# Las dos fuentes NO pueden resolver al mismo (carpeta, stem): comparten basename.
vistos = {}
for f in FUENTES:
    d = f["ckpt_dir"]
    llave = (str(d.resolve() if d.exists() else d), f["stem"])
    if llave in vistos:
        raise ValueError(
            f"Las fuentes '{vistos[llave]}' y '{f['clave']}' resuelven al mismo "
            f"{llave[0]}/{llave[1]}*.pt. Se compararía una corrida contra sí misma; "
            "revisá los YAML o los overrides de entorno."
        )
    vistos[llave] = f["clave"]

print(f"{'fuente':9s} {'num_steps':>10s} {'existe':>7s}  carpeta / log")
for f in FUENTES:
    ns = f"{f['num_steps']:,}" if f["num_steps"] else "—"
    print(f"{f['clave']:9s} {ns:>10s} {str(f['ckpt_dir'].is_dir()):>7s}  {f['ckpt_dir']}")
    print(f"{'':9s} {'':>10s} {'':>7s}  {f['log']}")

## 2. Descubrir los checkpoints de las dos corridas

Se descubren **todos** los de cada fuente y después se seleccionan los `N_CKPT_POR_FUENTE` más avanzados. Se descubre todo y
no solo lo seleccionado porque el solapamiento entre las dos corridas es la verificación de la sección 3, y para eso hace
falta el snapshot de 60k de `overfit` aunque no entre en la grilla de generación.

In [ ]:
# --- Descubrimiento por fuente ---
N_CKPT_POR_FUENTE = 4   # cuántos comparar de cada corrida (los más avanzados)
INCLUDE_RAW = False     # incluir el hermano de pesos crudos …_raw.pt del checkpoint final

Ckpt = namedtuple("Ckpt", "fuente paso ruta meta")


def discover_checkpoints(ckpt_dir, stem, fuente, *, include_raw=False):
    """Devuelve [Ckpt] de una corrida, ordenado por paso.

    Excluye los sidecars de resume (…_stepNNNNN.resume.pt), que matchean el glob pero NO son
    checkpoints de pesos. El paso sale de len(meta['history']) (un valor por paso), que también
    funciona para el checkpoint final (sin sufijo _stepNNNNN en el nombre).
    """
    ckpt_dir = pathlib.Path(ckpt_dir)
    if not ckpt_dir.is_dir():
        raise FileNotFoundError(
            f"Fuente '{fuente}': no existe la carpeta de checkpoints {ckpt_dir}"
        )

    encontrados = []
    for p in sorted(ckpt_dir.glob(f"{stem}*.pt")):
        if p.name.endswith(".resume.pt"):
            continue                                  # sidecar de reanudación, no pesos
        if p.stem.endswith("_raw") and not include_raw:
            continue
        try:
            _sd, meta = load_checkpoint(p, map_location="cpu")
        except (KeyError, RuntimeError) as exc:
            print(f"  (salteado {p.name}: no parece un checkpoint válido — {exc})")
            continue
        hist = meta.get("history") or []
        m = re.search(r"_step(\d+)", p.stem)
        paso = len(hist) if hist else (int(m.group(1)) if m else -1)
        encontrados.append(Ckpt(fuente, paso, p, meta))

    return sorted(encontrados, key=lambda c: c.paso)


TODOS, ckpts = [], []
for f in FUENTES:
    hallados = discover_checkpoints(f["ckpt_dir"], f["stem"], f["clave"],
                                    include_raw=INCLUDE_RAW)
    if not hallados:
        raise FileNotFoundError(
            f"Fuente '{f['clave']}': no encontré checkpoints '{f['stem']}*.pt' en "
            f"{f['ckpt_dir']}. ¿Está la corrida ahí, o hay que apuntar {f['env']}_CKPT?"
        )
    elegidos = hallados[-N_CKPT_POR_FUENTE:]
    seleccion = {(c.fuente, c.paso) for c in elegidos}
    TODOS.extend(hallados)
    ckpts.extend(elegidos)
    print(f"{f['clave']:9s} descubiertos {len(hallados):2d} | comparando {len(elegidos)}")
    for c in hallados:
        marca = "*" if (c.fuente, c.paso) in seleccion else " "
        print(f"   {marca} paso {c.paso:>7,} -> {c.ruta.name}")

# Orden final: por fuente (como están declaradas) y dentro de cada una por paso. Con los
# defaults salen los 8 pasos consecutivos 15k..120k, sin repetir ninguno.
orden = {f["clave"]: i for i, f in enumerate(FUENTES)}
ckpts.sort(key=lambda c: (orden[c.fuente], c.paso))

print(f"\n* = seleccionado. Total a comparar: {len(ckpts)} checkpoints")
print("  " + " -> ".join(f"{c.fuente}@{c.paso:,}" for c in ckpts))

por_paso_sel = {}
for c in ckpts:
    por_paso_sel.setdefault(c.paso, []).append(c.fuente)
choques = {p: fs for p, fs in por_paso_sel.items() if len(fs) > 1}
if choques:
    print("\nOJO: hay pasos que aparecen en más de una fuente SELECCIONADA:")
    for p, fs in sorted(choques.items()):
        print(f"  paso {p:,}: {fs} — son modelos de corridas distintas; se muestran los dos.")

## 3. Metadata, y la verificación de consistencia entre corridas

La tabla lleva una columna **fuente** porque los nombres de archivo son idénticos entre las dos, y una columna `sel` que marca
cuáles entran en la comparación.

Después, lo que sale gratis de tener las dos corridas: para cada paso presente en **ambas**, comparar sus números de
validación. Con el mismo `seed` y los mismos hiperparámetros deberían coincidir; lo que se aparte es el piso de
no-determinismo (orden de carga con `num_workers: 8`, AMP, `cudnn.benchmark`). Ese número es la resolución real con la que se
pueden leer las diferencias entre checkpoints: **una diferencia más chica que eso no significa nada.**

In [ ]:
# --- Tabla de metadata (todos los descubiertos, marcando los seleccionados) ---
sel = {(c.fuente, c.paso) for c in ckpts}
filas = []
for c in sorted(TODOS, key=lambda c: (orden[c.fuente], c.paso)):
    hist = c.meta.get("history") or []
    vh = c.meta.get("val_history") or []
    ultimo = vh[-1] if vh else {}
    filas.append({
        "sel": "*" if (c.fuente, c.paso) in sel else "",
        "fuente": c.fuente,
        "checkpoint": c.ruta.name,
        "paso": c.paso,
        "train (densa, últ.)": hist[-1] if hist else None,
        "val raw": ultimo.get("raw"),
        "val EMA": ultimo.get("ema"),
        "train fijo": ultimo.get("train"),
        "pesos": "EMA" if c.meta.get("ema") else "crudos",
    })

hdr = ["sel", "fuente", "checkpoint", "paso", "train (densa, últ.)", "val raw", "val EMA",
       "train fijo", "pesos"]


def fmt(v, h):
    if v is None:
        return "—"
    if isinstance(v, float):
        return f"{v:.6f}"
    return f"{v:,}" if h == "paso" else str(v)


anchos = [max(len(h), *(len(fmt(f[h], h)) for f in filas)) for h in hdr]
print(" | ".join(h.ljust(w) for h, w in zip(hdr, anchos)))
print("-+-".join("-" * w for w in anchos))
for f in filas:
    print(" | ".join(fmt(f[h], h).ljust(w) for h, w in zip(hdr, anchos)))

# La receta tiene que ser la MISMA en las dos corridas: la red es la variable de control.
recetas = {}
for c in TODOS:
    r = c.meta.get("model") or {}
    llave = (c.meta.get("sde_name"), str(c.meta.get("data_dim")), r.get("name"),
             r.get("score_parametrization"), json.dumps(r.get("kwargs"), sort_keys=True))
    recetas.setdefault(llave, []).append(f"{c.fuente}@{c.paso}")

print()
if len(recetas) == 1:
    (sde_n, dd, red_n, param, _kw), _ = next(iter(recetas.items()))
    print(f"Receta única en los {len(TODOS)} checkpoints descubiertos (comparación limpia):")
    print(f"  SDE: {sde_n} | data_dim={dd}")
    print(f"  Red: {red_n} | parametrización: {param or 'score directo'}")
else:
    print(f"OJO: hay {len(recetas)} recetas distintas entre los checkpoints. La comparación NO")
    print("es limpia — la red debería ser la variable de control. Detalle:")
    for llave, quienes in recetas.items():
        print(f"  sde={llave[0]} red={llave[2]} param={llave[3]} -> {quienes}")

# --- Gap de generalización por checkpoint ---
# El par comparable es 'train (examen fijo)' vs 'val (pesos vivos)': los dos se miden sobre
# las mismas 465 imágenes en orientación canónica, mismo t y mismo ruido. NO la curva densa.
print()
print("Gap de generalización val - train(examen fijo)  [+ = generaliza peor]:")
hubo_gap = False
for f in filas:
    if f["val raw"] is not None and f["train fijo"] is not None:
        hubo_gap = True
        print(f"  {f['fuente']:9s} paso {f['paso']:>7,}: {f['val raw'] - f['train fijo']:+.6f}")
if not hubo_gap:
    print("  (ningún checkpoint trae val_history con las dos series)")

# --- Verificación: los pasos presentes en AMBAS corridas deberían coincidir ---
por_paso = {}
for c in TODOS:
    por_paso.setdefault(c.paso, {})[c.fuente] = c
comunes = {p: d for p, d in por_paso.items() if len(d) > 1}

print()
if not comunes:
    print("No hay pasos presentes en las dos corridas: no se puede medir el piso de")
    print("no-determinismo. (Con keep_last_checkpoints=4 el solapamiento esperado es 60k.)")
else:
    print("Pasos presentes en las DOS corridas — mismo seed e hiperparámetros, así que la")
    print("diferencia es el piso de no-determinismo, no entrenamiento:")
    deltas = []
    for p, d in sorted(comunes.items()):
        claves = sorted(d, key=lambda k: orden[k])
        a, b = d[claves[0]], d[claves[1]]
        va = (a.meta.get("val_history") or [{}])[-1] or {}
        vb = (b.meta.get("val_history") or [{}])[-1] or {}
        print(f"  paso {p:,}  ({claves[0]} vs {claves[1]})")
        for k, etq in (("raw", "val raw"), ("ema", "val EMA"), ("train", "train fijo")):
            if va.get(k) is not None and vb.get(k) is not None:
                dif = vb[k] - va[k]
                deltas.append(abs(dif))
                rel = abs(dif) / abs(va[k]) if va[k] else float("nan")
                print(f"     {etq:10s} {va[k]:.6f} vs {vb[k]:.6f}   Δ={dif:+.6f} ({rel:.2%})")
    if deltas:
        print(f"\n  PISO DE NO-DETERMINISMO ≈ {max(deltas):.6f} (peor |Δ| observado).")
        print("  Diferencias entre checkpoints por debajo de eso no son interpretables.")
    else:
        print("  (los checkpoints comunes no traen val_history comparable)")

## 4. Las curvas de los dos experimentos

Dos figuras, y la segunda es la que contesta la pregunta del par 60k↔120k.

- **Figura 1** — un panel por corrida: la serie **densa** de entrenamiento (`event: "step"`, media móvil por cadencia de log)
  más las tres series **dispersas** de validación (`event: "val"`), con los pasos de los checkpoints seleccionados marcados.
  Comparten eje $y$, así que las alturas son comparables de un panel al otro.
- **Figura 2** — el **gap de generalización** `val(pesos vivos) − train(examen fijo)` contra el paso, las dos corridas en el
  mismo eje. Esta es la figura del sobreajuste: el gap creciendo es su definición operativa. La pérdida de validación sola no
  alcanza, porque puede bajar por mejora genuina mientras el gap se abre.

In [ ]:
# --- Parseo de los dos logs .jsonl ---
def load_train_log(path):
    """Lee el .jsonl y devuelve la lista de registros (ignora líneas truncadas)."""
    if path is None:
        return None
    path = pathlib.Path(path)
    if not path.exists():
        return None
    recs = []
    with open(path, "r", encoding="utf-8") as fh:
        for linea in fh:
            linea = linea.strip()
            if not linea:
                continue
            try:
                recs.append(json.loads(linea))
            except json.JSONDecodeError:
                continue    # línea truncada (corrida cortada): se ignora
    return recs


LOGS = {}
for f in FUENTES:
    recs = load_train_log(f["log"])
    if recs is None:
        print(f"{f['clave']:9s} sin log en {f['log']} -> se usa el val_history del meta")
        LOGS[f["clave"]] = {"steps": [], "vals": [], "fuente_datos": "meta"}
        continue
    steps_ev = [r for r in recs if r.get("event") == "step" and r.get("loss") is not None]
    vals_ev = [r for r in recs if r.get("event") == "val"]
    inicio = next((r for r in recs if r.get("event") == "start"), {})
    LOGS[f["clave"]] = {"steps": steps_ev, "vals": vals_ev, "fuente_datos": "jsonl"}
    extra = (f" | start: num_steps={inicio.get('num_steps')} amp={inicio.get('amp')}"
             if inicio else "")
    print(f"{f['clave']:9s} {len(recs):>6} registros | {len(steps_ev):>5} 'step' | "
          f"{len(vals_ev):>4} 'val'{extra}")
    devs = {r.get("device") for r in vals_ev if r.get("device")}
    if len(devs) > 1:
        print(f"  OJO ({f['clave']}): la validación se midió en más de un device {devs}. El "
              "examen fijo es específico del device, así que un salto puede ser del cambio "
              "de máquina y no del modelo.")


def puntos_val(clave, campo_log, campo_meta):
    """(pasos, valores) de una serie de validación: del .jsonl, o del meta como respaldo."""
    pts = [(r["step"], r[campo_log]) for r in LOGS[clave]["vals"]
           if r.get(campo_log) is not None]
    if pts:
        return list(zip(*pts))
    de_la_fuente = [c for c in TODOS if c.fuente == clave]
    if not de_la_fuente:
        return [], []
    vh = max(de_la_fuente, key=lambda c: c.paso).meta.get("val_history") or []
    pts = [(p["step"], p[campo_meta]) for p in vh if p.get(campo_meta) is not None]
    return list(zip(*pts)) if pts else ([], [])

In [ ]:
# --- Figura 1: un panel por corrida (eje y compartido) ---
SERIES = [("val_raw", "raw", "val (pesos vivos)", "C0"),
          ("val_ema", "ema", "val (EMA)", "C1"),
          ("train_fijo", "train", "train (examen fijo)", "C2")]

fig, axes = plt.subplots(1, len(FUENTES), figsize=(7.0 * len(FUENTES), 5), sharey=True)
axes = np.atleast_1d(axes)
todos_los_valores = []

for ax, f in zip(axes, FUENTES):
    clave = f["clave"]
    steps_ev = LOGS[clave]["steps"]
    if steps_ev:
        ax.plot([r["step"] for r in steps_ev], [r["loss"] for r in steps_ev],
                lw=0.9, alpha=0.7, color="0.45", label="train per-step (densa)")
        todos_los_valores += [r["loss"] for r in steps_ev]

    sufijo = "" if LOGS[clave]["fuente_datos"] == "jsonl" else " (meta)"
    for campo_log, campo_meta, etiqueta, color in SERIES:
        xs, ys = puntos_val(clave, campo_log, campo_meta)
        if not xs:
            continue
        ax.plot(xs, ys, marker="o", ms=3.5, lw=1.3, color=color, label=etiqueta + sufijo)
        todos_los_valores += list(ys)

    for c in ckpts:
        if c.fuente == clave:
            ax.axvline(c.paso, color="crimson", ls=":", lw=1.0, alpha=0.7)
    ax.plot([], [], color="crimson", ls=":", lw=1.0, label="checkpoints comparados")

    ns = f"{f['num_steps']:,}" if f["num_steps"] else "?"
    ax.set_xlabel("paso")
    ax.set_title(f"{clave} · {f['config']} · num_steps={ns}", fontsize=11)
    ax.grid(alpha=0.3)
    ax.legend(fontsize="small")

if todos_los_valores and min(todos_los_valores) > 0:
    axes[0].set_yscale("log")
axes[0].set_ylabel("pérdida DSM")
fig.suptitle("Pérdida de los dos experimentos — mismo eje y, escalas comparables",
             y=1.01, fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# --- Figura 2: el gap de generalización, las dos corridas en un eje ---
# gap = val(pesos vivos) - train(examen fijo). Los dos estimadores se miden sobre las mismas
# 465 imágenes en orientación canónica, con el mismo t y el mismo ruido, así que su
# diferencia es interpretable. El gap creciendo ES el sobreajuste.
fig, ax = plt.subplots(figsize=(9.5, 5))
hubo_datos = False
gaps = {}

for f in FUENTES:
    clave = f["clave"]
    xs_v, ys_v = puntos_val(clave, "val_raw", "raw")
    xs_t, ys_t = puntos_val(clave, "train_fijo", "train")
    pv, pt = dict(zip(xs_v, ys_v)), dict(zip(xs_t, ys_t))
    pasos_comunes = sorted(set(pv) & set(pt))
    if not pasos_comunes:
        print(f"{clave}: sin pasos con val Y train(fijo) medidos -> no hay gap para graficar")
        continue
    hubo_datos = True
    gap = [pv[p] - pt[p] for p in pasos_comunes]
    gaps[clave] = dict(zip(pasos_comunes, gap))
    etq = f"{clave} (num_steps={f['num_steps']:,})" if f["num_steps"] else clave
    ax.plot(pasos_comunes, gap, marker="o", ms=3.5, lw=1.4, color=f["color"], label=etq)
    for c in ckpts:
        if c.fuente == clave:
            ax.axvline(c.paso, color=f["color"], ls=":", lw=0.9, alpha=0.55)

if not hubo_datos:
    plt.close(fig)
    print("Ninguna corrida tiene las dos series: ¿faltó 'data.val_root' en los YAML?")
else:
    ax.axhline(0.0, color="0.3", lw=1.0)
    ax.set_xlabel("paso")
    ax.set_ylabel("val(pesos vivos) − train(examen fijo)")
    ax.set_title("Gap de generalización: creciente = sobreajuste\n"
                 "(punteadas verticales = pasos de los checkpoints comparados, por fuente)",
                 fontsize=11)
    ax.grid(alpha=0.3)
    ax.legend(fontsize="small")
    fig.tight_layout()
    plt.show()

    print("Gap en los pasos de los checkpoints seleccionados:")
    for c in ckpts:
        g = gaps.get(c.fuente, {}).get(c.paso)
        if g is not None:
            print(f"  {c.fuente:9s} paso {c.paso:>7,}: {g:+.6f}")

    for clave, serie in gaps.items():
        if len(serie) >= 2:
            pasos_ord = sorted(serie)
            print(f"  {clave}: gap {serie[pasos_ord[0]]:+.6f} (paso {pasos_ord[0]:,}) -> "
                  f"{serie[pasos_ord[-1]]:+.6f} (paso {pasos_ord[-1]:,})")

## 5. raw vs EMA, en las dos corridas

`val_raw` usa los **pesos vivos** (el último paso de Adam) y `val_ema` la **sombra EMA**. Con `ema_decay=0.999` la ventana
efectiva es de ~1000 pasos y la rampa de warmup satura cerca del paso 9000, así que al principio el EMA va **por encima**
(arrastra pesos viejos, peores) y después tiende a igualar o mejorar la curva cruda.

Una figura por corrida: arriba las dos curvas, abajo la diferencia $\text{val}_\text{EMA} - \text{val}_\text{raw}$
(**negativo = el EMA es mejor**).

In [ ]:
# --- Figura 3: raw vs EMA por corrida ---
for f in FUENTES:
    clave = f["clave"]
    xs_r, ys_r = puntos_val(clave, "val_raw", "raw")
    xs_e, ys_e = puntos_val(clave, "val_ema", "ema")
    pr, pe = dict(zip(xs_r, ys_r)), dict(zip(xs_e, ys_e))
    pasos_comunes = sorted(set(pr) & set(pe))
    if not pasos_comunes:
        print(f"{clave}: no hay puntos con raw Y ema. ¿La corrida tenía 'ema_decay' y "
              "'data.val_root'?")
        continue

    raw = [pr[p] for p in pasos_comunes]
    ema = [pe[p] for p in pasos_comunes]
    delta = [e - r for r, e in zip(raw, ema)]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(9, 6.5), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]}
    )
    ax1.plot(pasos_comunes, raw, marker="o", ms=4, lw=1.4, color="C0",
             label="val raw (pesos vivos)")
    ax1.plot(pasos_comunes, ema, marker="s", ms=4, lw=1.4, color="C1",
             label="val EMA (sombra)")
    for c in ckpts:
        if c.fuente == clave:
            ax1.axvline(c.paso, color="crimson", ls=":", lw=1.0, alpha=0.7)
    if min(raw + ema) > 0:
        ax1.set_yscale("log")
    ax1.set_ylabel("pérdida de validación")
    ax1.set_title(f"{clave} · pesos crudos vs sombra EMA "
                  f"(fuente: {LOGS[clave]['fuente_datos']})")
    ax1.grid(alpha=0.3)
    ax1.legend(fontsize="small")

    ancho = (max(1.0, 0.6 * (pasos_comunes[1] - pasos_comunes[0]))
             if len(pasos_comunes) > 1 else 1.0)
    ax2.bar(pasos_comunes, delta, width=ancho,
            color=["C2" if d < 0 else "C3" for d in delta])
    ax2.axhline(0.0, color="0.3", lw=1.0)
    ax2.set_xlabel("paso")
    ax2.set_ylabel("EMA − raw")
    ax2.grid(alpha=0.3)
    ax2.set_title("Diferencia (negativo = el EMA es mejor)", fontsize=10)
    fig.tight_layout()
    plt.show()

    mejor = sum(1 for d in delta if d < 0)
    print(f"{clave}: {len(delta)} puntos | EMA mejor en {mejor} | raw mejor en "
          f"{len(delta) - mejor}")
    print(f"  último (paso {pasos_comunes[-1]:,}): raw={raw[-1]:.6f} ema={ema[-1]:.6f} "
          f"delta={delta[-1]:+.6f}\n")

## 6. Generación comparada: los 8 checkpoints, mismo ruido

Los 8 generan con el **mismo sampler, el mismo presupuesto de NFE y la misma semilla**, así el $x_T$ inicial es idéntico y la
única variable es **cuánto entrenó** el modelo. Las filas van de menos a más entrenado, y el color de la etiqueta dice de qué
corrida viene cada una.

El presupuesto va en **NFE** (evaluaciones de la U-Net), no en pasos: `pc` gasta 2 por paso, así que 2000 NFE son 1000 pasos.
Es la misma convención del notebook 07, y así el número es comparable entre notebooks.

> **Costo:** son **8** corridas de sampleo completas, el doble que antes. Con la U-Net de 37M a 64×64 esto quiere GPU. La celda
> estima el total después del primer checkpoint; si no cierra, bajá `NFE_GEN`, `N_SAMPLES` o `N_CKPT_POR_FUENTE`.

In [ ]:
# --- Generación con los 8 checkpoints (mismo sampler + misma semilla) ---
SAMPLER = "pc"       # predictor-corrector: el más estable (ver notebooks 06/07)
N_SAMPLES = 4
NFE_GEN = 2000       # presupuesto en evaluaciones de la U-Net
N_STEPS_GEN = pasos_para(SAMPLER, NFE_GEN)
print(f"sampler '{SAMPLER}': {N_STEPS_GEN} pasos x {NFE_POR_PASO[SAMPLER]} = {NFE_GEN} NFE "
      f"| {len(ckpts)} checkpoints | device={DEVICE}")
if DEVICE == "cpu":
    print("AVISO: sin GPU. Son 8 sampleos de una U-Net de 37M a 64x64; en CPU no termina en")
    print("un tiempo razonable. Bajá NFE_GEN/N_SAMPLES o corré esto en la máquina del lab.")

# OJO: indexado por (fuente, paso). Con el paso solo, las dos corridas se pisarían: los
# pasos pueden repetirse entre fuentes y los nombres de archivo son idénticos.
muestras = {}
t_total = time.time()
for i, c in enumerate(ckpts):
    print(f"[{i+1}/{len(ckpts)}] {c.fuente}@{c.paso:,} ({c.ruta.name})…", flush=True)
    t0 = time.time()
    x0 = generate_from_checkpoint(
        c.ruta, sampler_name=SAMPLER, n_samples=N_SAMPLES, n_steps=N_STEPS_GEN,
        seed=SEED, device=DEVICE, **kwargs_para(SAMPLER),
    ).cpu()
    transcurrido = time.time() - t0
    muestras[(c.fuente, c.paso)] = x0
    print(f"    {transcurrido:6.1f}s  shape={tuple(x0.shape)} "
          f"rango=({float(x0.min()):.2f}, {float(x0.max()):.2f}) "
          f"finito={bool(torch.isfinite(x0).all())}")
    if i == 0 and len(ckpts) > 1:
        print(f"    (estimado para los {len(ckpts)}: "
              f"~{transcurrido * len(ckpts) / 60:.1f} min)")
print(f"total: {(time.time() - t_total) / 60:.1f} min")

color_de = {f["clave"]: f["color"] for f in FUENTES}
fig, axes = plt.subplots(
    len(ckpts), N_SAMPLES, figsize=(2.5 * N_SAMPLES, 2.7 * len(ckpts)), squeeze=False
)
for fila, c in enumerate(ckpts):
    arr = denorm(muestras[(c.fuente, c.paso)])
    for col in range(N_SAMPLES):
        ax = axes[fila][col]
        ax.imshow(arr[col])
        ax.axis("off")
        if col == 0:
            ax.set_title(f"{c.fuente} · paso {c.paso:,}", fontsize=10, fontweight="bold",
                         loc="left", color=color_de[c.fuente])
fig.suptitle(
    f"Mismo $x_T$ y mismo sampler ('{SAMPLER}', {N_STEPS_GEN} pasos = {NFE_GEN} NFE)\n"
    f"{len(ckpts)} checkpoints de {len(FUENTES)} corridas: efecto del entrenamiento",
    fontsize=12, y=1.005,
)
fig.tight_layout()
plt.show()

## Cierre

- **Dos fuentes, ocho checkpoints.** `base` (60k pasos) y `overfit` (120k) se descubren por separado, se etiquetan y se
  indexan por `(fuente, paso)` — obligatorio, porque las dos corridas escriben el mismo basename `cats_vp.pt` y un dict por
  paso las mezclaría en silencio. Con los defaults salen los 8 pasos consecutivos 15k…120k, sin repetir ninguno.
- **Las dos configs son la misma salvo el horizonte** (4 claves de 38: `num_steps` y las tres rutas de `out`), así que el eje
  de pasos es comparable sin normalizar nada y la corrida larga *prolonga* a la corta.
- **El solapamiento es una verificación gratis.** Los pasos presentes en ambas corridas deberían dar los mismos números; lo
  que se aparta es el piso de no-determinismo, y ese piso es la resolución real con la que se pueden leer las diferencias
  entre checkpoints.
- **La figura del sobreajuste es el gap**, no la pérdida: `val(pesos vivos) − train(examen fijo)` contra el paso, las dos
  corridas superpuestas. La validación sola puede bajar mientras el gap se abre.
- **raw vs EMA** se grafica por corrida, con el panel de diferencia para ver de un vistazo cuándo el promediado ayuda.
- **La grilla de generación** aísla el efecto del entrenamiento: mismo ruido inicial, mismo sampler y mismo NFE en las 8 filas.

Para el estudio del **Eje 2** (los 4 samplers sobre *un* checkpoint, a NFE igualado) el notebook es el `07`.